# 05 Local and Global Graph Search

Builds on the Neo4j graph from notebook 04.

- Local retrieval: property, building, and community context.
- Community analysis: counts and price statistics for one community.
- Global retrieval: portfolio-wide statistics and comparisons.

## 1. Connect to Neo4j

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

ROOT = Path.cwd()
if not (ROOT / "notebooks").exists() and (ROOT.parent / "notebooks").exists():
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv(
    "NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j")
)
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

if not NEO4J_URI or not NEO4J_PASSWORD:
    raise ValueError("Configure NEO4J_URI and NEO4J_PASSWORD in .env")

driver = GraphDatabase.driver(
    NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print("Connected to Neo4j.")


def run_query(cypher, params=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        return [r.data() for r in session.run(cypher, params or {})]

Connected to Neo4j.


## 2. Local retrieval

Property IDs are looked up exactly. Community retrieval follows the
existing LOCATED_IN, IN_BUILDING, and HAS_TYPE relationships.

In [11]:
def property_context(property_id):
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property {property_id: $property_id})
        OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
        OPTIONAL MATCH (p)-[:IN_BUILDING]->(b:Building)
        OPTIONAL MATCH (p)-[:HAS_TYPE]->(t:PropertyType)
        RETURN p.property_id AS property_id,
               p.title AS title,
               p.description AS description,
               p.price AS price,
               p.beds AS beds,
               p.bathrooms AS bathrooms,
               p.area AS area,
               p.listing_type AS listing_type,
               p.status AS status,
               coalesce(c.name, p.community) AS community,
               coalesce(b.name, p.building) AS building,
               coalesce(t.name, p.property_type) AS property_type,
               p.address AS address,
               p.url AS url
        ''',
        {"property_id": str(property_id)}
    ))


def community_properties(community, limit=10):
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)-[:LOCATED_IN]->(c:Community)
        WHERE c.name IS NOT NULL
            AND toLower(toString(c.name)) CONTAINS toLower($community)
        OPTIONAL MATCH (p)-[:IN_BUILDING]->(b:Building)
        OPTIONAL MATCH (p)-[:HAS_TYPE]->(t:PropertyType)
        RETURN p.property_id AS property_id,
               p.title AS title,
               p.price AS price,
               p.beds AS beds,
               p.bathrooms AS bathrooms,
               p.listing_type AS listing_type,
               p.status AS status,
               c.name AS community,
               coalesce(b.name, p.building) AS building,
               coalesce(t.name, p.property_type) AS property_type,
               p.url AS url
        ORDER BY p.price
        LIMIT $limit
        ''',
        {"community": community, "limit": int(limit)}
    ))


def community_summary(community):
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)-[:LOCATED_IN]->(c:Community)
        WHERE c.name IS NOT NULL
            AND toLower(toString(c.name)) CONTAINS toLower($community)
        WITH p, c, toFloat(coalesce(p.price, 0)) AS price
        RETURN c.name AS community,
               coalesce(p.listing_type, 'unknown') AS listing_type,
               count(p) AS property_count,
               round(avg(CASE WHEN price > 0 THEN price ELSE NULL END))
                   AS average_price,
               min(CASE WHEN price > 0 THEN price ELSE NULL END)
                   AS minimum_price,
               max(CASE WHEN price > 0 THEN price ELSE NULL END)
                   AS maximum_price
        ORDER BY property_count DESC
        ''',
        {"community": community}
    ))

## 3. Global retrieval

These aggregations deliberately group by listing_type. Sale prices and
rental prices should not be treated as equivalent measurements.

In [12]:
def portfolio_summary():
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)
        WITH p, toFloat(coalesce(p.price, 0)) AS price
        RETURN coalesce(p.listing_type, 'unknown') AS listing_type,
               count(p) AS property_count,
               count(DISTINCT p.community) AS communities,
               count(DISTINCT p.building) AS buildings,
               round(avg(CASE WHEN price > 0 THEN price ELSE NULL END))
                   AS average_price,
               min(CASE WHEN price > 0 THEN price ELSE NULL END)
                   AS minimum_price,
               max(CASE WHEN price > 0 THEN price ELSE NULL END)
                   AS maximum_price
        ORDER BY property_count DESC
        '''
    ))


def compare_communities(limit=20):
    return pd.DataFrame(run_query(
        '''
        MATCH (p:Property)
        WHERE p.community IS NOT NULL
        WITH p, toFloat(coalesce(p.price, 0)) AS price
        RETURN p.community AS community,
               coalesce(p.listing_type, 'unknown') AS listing_type,
               count(p) AS property_count,
               round(avg(CASE WHEN price > 0 THEN price ELSE NULL END))
                   AS average_price,
               min(CASE WHEN price > 0 THEN price ELSE NULL END)
                   AS minimum_price,
               max(CASE WHEN price > 0 THEN price ELSE NULL END)
                   AS maximum_price
        ORDER BY property_count DESC
        LIMIT $limit
        ''',
        {"limit": int(limit)}
    ))


# Examples: uncomment after connecting.
# display(property_context("4383"))
# display(portfolio_summary())
# display(compare_communities())

In [13]:
display(community_properties("Dubai Marina"))

,property_id,title,price,beds,bathrooms,listing_type,status,community,building,property_type,url
0,2720,apartment to rent,38000.0,0,1,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
1,3655,apartment to rent,43000.0,0,0,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
2,4476,1 bedroom apartment to rent,45000.0,1,2,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
3,3368,1 bedroom apartment to rent,50000.0,1,1,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
4,3654,apartment to rent,50000.0,0,1,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
5,3627,1 bedroom apartment to rent,50000.0,1,1,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
6,3369,apartment to rent,53000.0,0,1,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
7,141185,apartment to rent,53000.0,0,1,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
8,150918,apartment to rent,53000.0,0,1,rental,To Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...
9,3641,1 bedroom apartment to rent,55000.0,1,2,rental,Let,Dubai Marina,"[""Apartment""]",apartment,https://www.hausandhaus.com/properties/to-rent...


In [14]:
display(community_summary("Dubai Marina"))

,community,listing_type,property_count,average_price,minimum_price,maximum_price
0,Dubai Marina,rental,441,160342.0,38000.0,1799999.0
1,Dubai Marina,purchase,110,4035580.0,845000.0,40000000.0


## 4. Choosing a strategy

| Question | Strategy |
|---|---|
| Tell me about property 4383 | Exact lookup + local graph |
| What is available in Dubai Marina? | Community retrieval, optionally combined with Milvus |
| What is the average price in Dubai Marina? | Community aggregation |
| Compare communities across the portfolio | Global aggregation |

Close the driver when finished.

In [15]:

driver.close()